# Vector Operations and Fields

From the algebra of dot and cross products to the calculus of divergence and curl — interactive 3-D visualizations of the mathematical tools that underpin electromagnetism and fluid dynamics.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider
from mpl_toolkits.mplot3d import Axes3D
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
%matplotlib inline



## Dot Product and Cross Product

Given two vectors $\vec{v}_1$ and $\vec{v}_2$:

**Dot product** (scalar):

$$\vec{v}_1 \cdot \vec{v}_2 = |\vec{v}_1|\,|\vec{v}_2|\,\cos\theta$$

It measures how much two vectors point in the same direction. Zero when perpendicular, negative when opposing.

**Cross product** (vector, 3-D only):

$$\vec{v}_1 \times \vec{v}_2 = |\vec{v}_1|\,|\vec{v}_2|\,\sin\theta\;\hat{n}$$

The result is perpendicular to both inputs (right-hand rule). Its magnitude equals the area of the parallelogram they span. Zero when parallel.

| Angle | Dot product | Cross product magnitude |
|---|---|---|
| $0°$ | max ($+1$) | $0$ |
| $90°$ | $0$ | max ($1$) |
| $180°$ | min ($-1$) | $0$ |

Sweep the angle slider and observe all three vectors in real time.


In [2]:
@interact(
    angle=FloatSlider(min=0, max=360, step=1, value=45, description='Angle (°)')
)
def plot_dot_cross(angle):
    angle_rad = np.deg2rad(angle)
    v1 = np.array([1, 0, 0])
    v2 = np.array([np.cos(angle_rad), np.sin(angle_rad), 0])
    dot_val = np.dot(v1, v2)
    cross_val = np.cross(v1, v2)

    fig = plt.figure(figsize=(7, 6))
    ax = fig.add_subplot(111, projection='3d')
    ax.quiver(0, 0, 0, *v1, color='r', lw=2.5, arrow_length_ratio=0.12, label='v₁')
    ax.quiver(0, 0, 0, *v2, color='b', lw=2.5, arrow_length_ratio=0.12, label='v₂')
    ax.quiver(0, 0, 0, *cross_val, color='g', lw=2.5, arrow_length_ratio=0.12, label='v₁ × v₂')

    # Angle arc
    arc_r = 0.5
    arc_t = np.linspace(0, angle_rad, 60)
    ax.plot(arc_r*np.cos(arc_t), arc_r*np.sin(arc_t), np.zeros_like(arc_t), 'k-', lw=1.5)
    mid = angle_rad / 2
    ax.text(0.55*np.cos(mid), 0.55*np.sin(mid), 0, f'{angle:.0f}°', fontsize=10)

    ax.text2D(0.02, 0.95, f'Dot:   v₁·v₂ = {dot_val:.3f}', transform=ax.transAxes, fontsize=11, color='purple')
    ax.text2D(0.02, 0.90, f'Cross: |v₁×v₂| = {np.linalg.norm(cross_val):.3f}', transform=ax.transAxes, fontsize=11, color='green')
    ax.set_xlim(-1.2, 1.2); ax.set_ylim(-1.2, 1.2); ax.set_zlim(-1.2, 1.2)
    ax.set_xlabel('X'); ax.set_ylabel('Y'); ax.set_zlabel('Z')
    ax.set_title('Dot & Cross Product')
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.show()


interactive(children=(FloatSlider(value=45.0, description='Angle (°)', max=360.0, step=1.0), Output()), _dom_c…


## Gravitational Vector Field

A point mass $M$ creates a gravitational field

$$\vec{g}(\vec{r}) = -\frac{GM}{|\vec{r}-\vec{r}_0|^2}\,\hat{r}$$

At every point in space the field vector points **toward** the mass (attractive). The magnitude falls off as $1/r^2$, so arrows close to the source are much larger than those far away. This is the same inverse-square structure shared by the electric field of a point charge.

Move the source mass around the plane and watch the entire field rearrange.


In [3]:
@interact(
    cx=FloatSlider(min=-4, max=4, step=0.5, value=0, description='Source X'),
    cy=FloatSlider(min=-4, max=4, step=0.5, value=0, description='Source Y')
)
def plot_grav_field(cx, cy):
    x = np.linspace(-5, 5, 18)
    y = np.linspace(-5, 5, 18)
    X, Y = np.meshgrid(x, y)
    G_scaled, mass = 6.674, 1
    dist = np.sqrt((X - cx)**2 + (Y - cy)**2)
    mask = dist < 1.2
    F_mag = G_scaled * mass / dist**2
    F_mag[mask] = 0
    dx = -(X - cx) / dist
    dy = -(Y - cy) / dist
    dx[mask] = 0; dy[mask] = 0

    fig, ax = plt.subplots(figsize=(6, 6))
    ax.quiver(X, Y, F_mag*dx, F_mag*dy, np.log1p(F_mag), cmap='inferno_r', scale=25)
    circle = plt.Circle((cx, cy), 0.6, color='royalblue', zorder=5)
    ax.add_patch(circle)
    ax.annotate('M', (cx, cy), ha='center', va='center', fontsize=14, color='white', fontweight='bold')
    ax.set_xlim(-5, 5); ax.set_ylim(-5, 5)
    ax.set_aspect('equal')
    ax.set_xlabel('X'); ax.set_ylabel('Y')
    ax.set_title('Gravitational Field  g(r) ∝ 1/r²')
    ax.grid(True, alpha=0.2)
    plt.tight_layout()
    plt.show()


interactive(children=(FloatSlider(value=0.0, description='Source X', max=4.0, min=-4.0, step=0.5), FloatSlider…


## Divergence and Curl

Two fundamental differential operators act on vector fields:

### Divergence (scalar output)

$$\nabla \cdot \vec{F} = \frac{\partial F_x}{\partial x} + \frac{\partial F_y}{\partial y} + \frac{\partial F_z}{\partial z}$$

Divergence measures the net *outflow* of a field at a point. A positive divergence means the field is "spreading out" — like the electric field from a positive charge. The **divergence theorem** (Gauss) relates the volume integral of divergence to the surface flux:

$$\oint_S \vec{F}\cdot d\vec{A} = \int_V (\nabla\cdot\vec{F})\,dV$$

### Curl (vector output)

$$\nabla \times \vec{F} = \left(\frac{\partial F_z}{\partial y}-\frac{\partial F_y}{\partial z}\right)\hat{x} + \cdots$$

Curl measures local *rotation* of the field. A non-zero curl means the field has a "swirl" — like the magnetic field around a current-carrying wire. **Stokes' theorem** relates the surface integral of curl to the line integral around the boundary:

$$\oint_C \vec{F}\cdot d\vec{l} = \int_S (\nabla\times\vec{F})\cdot d\vec{A}$$

Below you can adjust the source strength (diverging radial field) and vortex strength (rotating tangential field) and see both theorems in action.


In [4]:
def radial_field(x, y, z, s=1.0):
    r = np.sqrt(x**2 + y**2 + z**2)
    f = s / (r**3 + 0.1)
    return f*x, f*y, f*z

@interact(
    strength=FloatSlider(min=0.5, max=3, step=0.1, value=1.5, description='Source str.')
)
def plot_divergence(strength):
    fig = plt.figure(figsize=(7, 6))
    ax = fig.add_subplot(111, projection='3d')
    cube = 1.5
    half = cube / 2

    # Draw transparent cube edges
    for s1 in [-1, 1]:
        for s2 in [-1, 1]:
            ax.plot([s1*half, s1*half], [s2*half, s2*half], [-half, half], 'b-', alpha=0.3)
            ax.plot([s1*half, s1*half], [-half, half], [s2*half, s2*half], 'b-', alpha=0.3)
            ax.plot([-half, half], [s1*half, s1*half], [s2*half, s2*half], 'b-', alpha=0.3)

    ax.scatter(0, 0, 0, color='red', s=200, edgecolors='darkred', lw=2, zorder=5, label='Source +Q')
    n = 4
    pts = np.linspace(-half, half, n)
    for face_val in [-half, half]:
        for a in pts:
            for b in pts:
                for (x, y, z) in [(face_val, a, b), (a, face_val, b), (a, b, face_val)]:
                    fx, fy, fz = radial_field(x, y, z, strength)
                    sc = 0.15
                    ax.quiver(x, y, z, fx*sc, fy*sc, fz*sc, color='steelblue', alpha=0.7, arrow_length_ratio=0.3, lw=1.2)

    # Approximate surface flux
    flux = 0
    delta = cube / (n - 1)
    area = delta**2
    for fv in [-half, half]:
        sgn = 1 if fv > 0 else -1
        for a in pts:
            for b in pts:
                _, _, fz = radial_field(a, b, fv, strength)
                flux += sgn * fz * area

    ax.set_xlabel('X'); ax.set_ylabel('Y'); ax.set_zlabel('Z')
    ax.set_xlim(-1.2, 1.2); ax.set_ylim(-1.2, 1.2); ax.set_zlim(-1.2, 1.2)
    ax.set_title(f'Divergence Theorem — Surface Flux ≈ {flux:.2f}')
    ax.text2D(0.02, 0.92, '∫∫ F·dA = ∫∫∫ (∇·F) dV', transform=ax.transAxes, fontsize=11, color='navy')
    ax.legend(fontsize=9)
    ax.view_init(elev=22, azim=40)
    plt.tight_layout()
    plt.show()


interactive(children=(FloatSlider(value=1.5, description='Source str.', max=3.0, min=0.5), Output()), _dom_cla…

In [5]:
def vortex_field(x, y, z, s=1.0):
    r = np.sqrt(x**2 + y**2)
    f = s / (r + 0.1)
    return -y*f, x*f, 0

@interact(
    strength=FloatSlider(min=0.5, max=3, step=0.1, value=1.5, description='Vortex str.')
)
def plot_curl(strength):
    fig = plt.figure(figsize=(7, 6))
    ax = fig.add_subplot(111, projection='3d')
    loop_r = 0.8

    # Boundary loop
    theta = np.linspace(0, 2*np.pi, 100)
    ax.plot(loop_r*np.cos(theta), loop_r*np.sin(theta), np.zeros_like(theta), 'b-', lw=3, label='Loop C')

    # Surface disk
    r_d = np.linspace(0, loop_r, 10)
    t_d = np.linspace(0, 2*np.pi, 40)
    R, T = np.meshgrid(r_d, t_d)
    ax.plot_surface(R*np.cos(T), R*np.sin(T), np.zeros_like(R), alpha=0.15, color='lightgreen')

    ax.scatter(0, 0, 0, color='purple', s=200, marker='x', lw=3, label='Vortex centre')

    # Field vectors
    for ri in np.linspace(0.25, 1.2, 4):
        for t in np.linspace(0, 2*np.pi, 8, endpoint=False):
            x, y = ri*np.cos(t), ri*np.sin(t)
            fx, fy, fz = vortex_field(x, y, 0, strength)
            sc = 0.18
            ax.quiver(x, y, 0, fx*sc, fy*sc, 0, color='red', alpha=0.7, arrow_length_ratio=0.3, lw=1.3)

    # Curl arrows (z-direction)
    for i in range(6):
        ang = 2*np.pi*i/6
        rx, ry = 0.4*np.cos(ang), 0.4*np.sin(ang)
        r_pt = np.sqrt(rx**2 + ry**2)
        curl_z = 2*strength / (r_pt + 0.1) * 0.08
        ax.quiver(rx, ry, 0, 0, 0, curl_z, color='green', lw=2.5, arrow_length_ratio=0.25, label='∇×F' if i == 0 else '')

    # Line integral
    n_loop = 100
    line_int = 0
    for i in range(n_loop):
        t = 2*np.pi*i/n_loop
        x, y = loop_r*np.cos(t), loop_r*np.sin(t)
        dx = -loop_r*np.sin(t)*(2*np.pi/n_loop)
        dy = loop_r*np.cos(t)*(2*np.pi/n_loop)
        fx, fy, _ = vortex_field(x, y, 0, strength)
        line_int += fx*dx + fy*dy

    ax.set_xlabel('X'); ax.set_ylabel('Y'); ax.set_zlabel('Z')
    ax.set_xlim(-1.5, 1.5); ax.set_ylim(-1.5, 1.5); ax.set_zlim(-0.6, 0.6)
    ax.set_title(f"Stokes' Theorem — Circulation ≈ {line_int:.2f}")
    ax.text2D(0.02, 0.92, '∮ F·dl = ∫∫ (∇×F)·dA', transform=ax.transAxes, fontsize=11, color='navy')
    handles, labels = ax.get_legend_handles_labels()
    by_label = dict(zip(labels, handles))
    ax.legend(by_label.values(), by_label.keys(), fontsize=8, loc='upper right')
    ax.view_init(elev=28, azim=40)
    plt.tight_layout()
    plt.show()


interactive(children=(FloatSlider(value=1.5, description='Vortex str.', max=3.0, min=0.5), Output()), _dom_cla…